# Chapter 04-01 · Framing: turning a request into a task

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** easy to read, hard to do

**Prerequisites:** 02-01 for what a row is, 02-08 for the applied workflow, 03-01 for summaries.

**Position in the learning path:** module 04, chapter 1 of 8. This module is the spine of the course.

---

## Why this matters

Module 03 finished the mathematics. You can fit a model. This module is about everything that decides
whether fitting one was worth doing, and it starts with the step that is skipped most often.

Somebody says: **"predict which members are going to leave, so we can send them an offer."**

That is a request, not a task. It has no unit, no target, no moment at which the prediction is made, and
no statement of what is known at that moment. Four different competent people will build four different
models from it, all correctly, and get numbers that disagree by a factor of fifteen - and none of them
will be wrong, because the question did not say.

This chapter turns that sentence into a **prediction contract**: five questions with written answers,
settled before any data is loaded. It is the cheapest hour in the entire course. The alternative is
discovering in month three that the model predicts something nobody wanted.

## What you will be able to do

- Ask the five framing questions, and recognise a request that has not answered them
- Show that "the churn rate" has at least three defensible values, and compute them
- Draw the timeline of a prediction task: feature window, prediction point, horizon
- Say why a horizon must be chosen before a model is built, not after
- Spot a column that will not exist at prediction time, and price the mistake
- Write a contract another person could build from

## Warm-up: retrieve, do not reread

1. What does "one row is one what?" decide about every average you compute afterwards?
2. In 03-08, what did a falling loss curve *not* prove?
3. If a class is 13% of the data, what accuracy does a model that predicts "no" for everyone achieve?

<br>

*Answers: (1) everything - the unit sets what a mean is a mean of, and a per-row average weights busy
entities more than quiet ones. (2) that the fit had arrived; only comparison against an achievable value
shows that. (3) 87%.*

## The data

A gym chain, 24 months, one row per **member-month**. It is synthetic, and deliberately raw: this is the
shape data arrives in, before anybody has decided what a row of the *modelling* table should be.

| Column | Meaning |
|---|---|
| `member_id` | the member |
| `month` | 1 to 24 |
| `visits` | visits that month |
| `tickets` | support contacts that month |
| `cancelled` | 1 if the membership ended in that month, and the member has no later rows |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# SYNTHETIC. One row per member-month at a gym chain, 24 months.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)          # unobserved: how attached the member is
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)   # a slow random walk away from the gym
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members()
print("%d rows, %d members, months %d to %d"
      % (len(panel), panel.member_id.nunique(), panel.month.min(), panel.month.max()))
print(panel.head(4).to_string(index=False))

## Question 1 · What is one row?

The request said "which members are going to leave". So the unit is a member - obviously.

Except the data is member-months, and the moment somebody asks "what is our churn rate?" the ambiguity
becomes a number. Here are three answers, all computed correctly from the same table.

**Predict before running:** how far apart will they be?

In [ ]:
members = panel.member_id.nunique()
ever_cancelled = panel.groupby("member_id").cancelled.max()
month_12 = panel[panel.month == 12]

rates = pd.DataFrame([
    {"unit": "one member",
     "question it answers": "what share of members ever leave?",
     "numerator": int(ever_cancelled.sum()), "denominator": members,
     "rate": "%.2f%%" % (100 * ever_cancelled.mean())},
    {"unit": "one member-month",
     "question it answers": "in a given month, what share of memberships end?",
     "numerator": int(panel.cancelled.sum()), "denominator": len(panel),
     "rate": "%.2f%%" % (100 * panel.cancelled.mean())},
    {"unit": "one member, one month",
     "question it answers": "of those with us in month 12, how many left that month?",
     "numerator": int(month_12.cancelled.sum()), "denominator": len(month_12),
     "rate": "%.2f%%" % (100 * month_12.cancelled.mean())},
])
print(rates.to_string(index=False))
print()
print("largest divided by smallest: %.1f times"
      % (ever_cancelled.mean() / month_12.cancelled.mean()))

**33.00%, 2.19% and 1.51% - a factor of 22 between the ends.** Nobody has made a mistake. They are
answers to three different questions, and the word "churn rate" points at all three.

This matters beyond the reporting. **The unit decides what a row of your modelling table is**, and that
decides what the model can be asked at prediction time:

- one row per **member** - one prediction per person, ever. Fine for "who should get the offer" and
  useless for "when".
- one row per **member-month** - a prediction every month for every active member. This is what an
  operational retention programme actually needs, and it is 9,059 rows rather than 600.

The second is almost always the right choice for this kind of problem, and it is almost never what the
first draft of the request implies.

**The rule to take away:** a rate is uninterpretable without its denominator, and the denominator is the
unit of observation. If somebody tells you a churn rate and you do not know what one row was, you have
not been told anything.

In [ ]:
tenure = panel.groupby("member_id").month.agg(["min", "max"])
tenure["months"] = tenure["max"] - tenure["min"] + 1

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

labels = ["per member\n(ever)", "per member-month", "per member\n(month 12 only)"]
values = [100 * ever_cancelled.mean(), 100 * panel.cancelled.mean(), 100 * month_12.cancelled.mean()]
bars = left.bar(labels, values, color=["#0072B2", "#D55E00", "#009E73"])
for bar, value in zip(bars, values):
    left.text(bar.get_x() + bar.get_width() / 2, value + 0.6, "%.2f%%" % value, ha="center")
left.set_ylabel("'the churn rate'")
left.set_ylim(0, 38)
left.set_title("One phrase, three numbers, no mistakes")

right.hist(tenure["months"], bins=range(1, 26), color="#999999", edgecolor="white")
right.axvline(tenure["months"].median(), color="#D55E00", linestyle="--")
right.text(1.5, 46, "median %.0f months" % tenure["months"].median(), color="#D55E00")
right.set_xlabel("months of membership observed")
right.set_ylabel("members")
right.set_title("Why the units differ: memberships vary in length", fontsize=11)

plt.tight_layout()
plt.show()

The right-hand panel is the mechanism. A member who stays 24 months contributes 24 member-month rows and
one member row. **Averaging over member-months therefore weights long memberships more heavily**, and
averaging over members weights everybody equally. Neither is wrong; they are answers to different
questions, and the histogram is why the answers cannot agree.

This is 02-01's lesson arriving where it does damage.

## Question 2 · What exactly is the target?

"Going to leave" is not a column. Making it one requires three decisions that the request did not make:

- **What counts as leaving?** A cancellation is easy here. In real data it is usually a choice between
  an explicit cancellation, a lapsed payment, a period of no activity, or a downgrade - and these give
  different populations.
- **Leaving by when?** Question 4.
- **Leaving relative to what moment?** Question 3.

The target is a **column you construct**, and constructing it is a modelling decision, not data cleaning.
Write down the line of code that builds it, because that line *is* the definition, and it is the thing
that will be misremembered in three months.

## Question 3 · When is the prediction made?

This is the question that separates a task from a wish, and it is the one beginners skip entirely.

A prediction is made at a **moment**. Before that moment, data exists and may be used. After it, data
exists in your table and **may not be used**, because in production it will not have happened yet.

Pick month 12 as the prediction point. Everything from months 1 to 12 is a legitimate feature.
Everything from month 13 onward is the future.

In [ ]:
CUT = 12          # the prediction point: end of month 12
HORIZON = 6       # we predict cancellation within the next 6 months

active_at_cut = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active_at_cut) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active_at_cut) & (panel.month > CUT)]

# the target: did this member cancel in months 13 to 18?
cancelled_in_window = (future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)]
                       .member_id.unique())
target = pd.Series(np.isin(active_at_cut, cancelled_in_window).astype(int), index=active_at_cut)

print("members active at the end of month %d : %d" % (CUT, len(active_at_cut)))
print("of those, cancelled in months %d-%d    : %d  (%.2f%%)"
      % (CUT + 1, CUT + HORIZON, target.sum(), 100 * target.mean()))
print()
print("rows the model may see  : months 1-%d" % CUT)
print("rows that define the answer: months %d-%d" % (CUT + 1, CUT + HORIZON))
print("rows that are neither     : months %d-24, discarded" % (CUT + HORIZON + 1))

Notice what that cell threw away. Members who had already cancelled by month 12 are **not** in the
population - you cannot predict the churn of somebody who has already left. And months 19 to 24 are
discarded, because they are outside the window the target is defined on.

**A framing decision is mostly a decision about which rows do not participate.** That is why it has to
come first: it determines the dataset, and the dataset determines everything after.

Here it is as a picture, which is the form worth remembering.

In [ ]:
np.random.seed(0)
sample = panel[panel.member_id.isin(active_at_cut[:9])]
fig, ax = plt.subplots(figsize=(11, 5))

ax.axvspan(0.5, CUT + 0.5, color="#0072B2", alpha=0.10)
ax.axvspan(CUT + 0.5, CUT + HORIZON + 0.5, color="#D55E00", alpha=0.12)
ax.axvspan(CUT + HORIZON + 0.5, 24.5, color="#999999", alpha=0.10)

for row, member in enumerate(active_at_cut[:9]):
    span = panel[panel.member_id == member]
    ax.plot([span.month.min(), span.month.max()], [row, row], color="#444444", linewidth=1.2)
    ax.plot(span.month, np.full(len(span), row), "o", color="#444444", markersize=2.5)
    if span.cancelled.max() == 1:
        end = span[span.cancelled == 1].month.iloc[0]
        ax.plot(end, row, "X", color="#D55E00", markersize=11)

ax.axvline(CUT + 0.5, color="#000000", linewidth=2)
ax.text(CUT + 0.35, 9.3, "prediction made here", ha="right", fontsize=10)
ax.text(6.5, 9.3, "features: months 1-12", ha="center", color="#0072B2", fontsize=10)
ax.text(CUT + HORIZON / 2 + 0.5, 9.3, "answer: months 13-18", ha="center", color="#D55E00", fontsize=10)
ax.text(21.5, 9.3, "discarded", ha="center", color="#666666", fontsize=10)

ax.set_yticks(range(9))
ax.set_yticklabels(["member %d" % m for m in active_at_cut[:9]], fontsize=8)
ax.set_xlabel("month")
ax.set_xlim(0.5, 24.5)
ax.set_ylim(-0.8, 10.0)
ax.set_title("The shape of every prediction task: a wall, a window before it, a window after it")
plt.tight_layout()
plt.show()

**That black line is the whole chapter.** Draw it for any supervised problem you are handed and most of
the framing questions answer themselves:

- Anything to the left of the wall is a feature.
- Anything to the right of it is the answer, or is discarded.
- **A column that crosses the wall is a bug**, however good it makes the model look. That is Question 5.

The picture also shows why "predict churn" was underspecified: move the wall to month 6 and you get a
different population, a different target and different features from the same table, with nothing to say
that either version is the wrong one.

## Question 4 · How far ahead? The horizon

The wall says *when*. The horizon says *how far past it* the answer looks. It cannot be left to the
model, and it changes the problem completely.

In [ ]:
rows = []
for horizon in [1, 2, 3, 6, 9, 12]:
    left = (future[(future.month <= CUT + horizon) & (future.cancelled == 1)].member_id.nunique())
    rows.append({"horizon (months)": horizon,
                 "members who cancel": left,
                 "base rate": "%.2f%%" % (100 * left / len(active_at_cut)),
                 "accuracy of 'nobody leaves'": "%.2f%%" % (100 * (1 - left / len(active_at_cut)))})
horizons = pd.DataFrame(rows)
print(horizons.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
months = np.arange(1, 13)
cumulative = [100 * future[(future.month <= CUT + h) & (future.cancelled == 1)].member_id.nunique()
              / len(active_at_cut) for h in months]
ax.plot(months, cumulative, "o-", color="#0072B2")
for h in (1, 6, 12):
    ax.annotate("%.2f%%" % cumulative[h - 1], (h, cumulative[h - 1]),
                textcoords="offset points", xytext=(6, -12), color="#D55E00")
ax.set_xlabel("horizon: months after the prediction point")
ax.set_ylabel("share of members who cancel (%)")
ax.set_title("The same members, the same data. The horizon alone moves the answer 13-fold")
plt.tight_layout()
plt.show()

**1.73% at one month, 13.44% at six, 22.84% at twelve.** Same members, same wall, same features. The
horizon alone moves the target by a factor of thirteen.

Three consequences, and the third is the one that bites:

1. **The metric moves with it.** A model reporting 98% accuracy at a one-month horizon has beaten
   nothing - predicting "nobody leaves" scores 98.27%. At twelve months the same trick scores 77.16%.
   **You cannot compare two churn models without knowing their horizons**, and vendors' numbers rarely
   state one.
2. **The features that matter move with it.** A one-month horizon is dominated by whatever is happening
   right now; a twelve-month horizon is dominated by stable traits. These are different models.
3. **The horizon has to be chosen from the decision, not the data.** If the retention team can only run
   an offer campaign quarterly, a one-month horizon produces predictions nobody can act on, and a
   twelve-month one produces a list too long to fund. **The right horizon is the one that matches the
   time it takes to do something about it.**

That last point is worth stating in general, because it is the reason framing precedes modelling:
**every one of these five questions is answered by the decision the prediction will feed, not by the
data.** The data cannot tell you what somebody plans to do with the answer.

## Question 5 · What is actually known at prediction time?

This is where framing stops being paperwork and starts saving projects.

Every column in your table exists *now*, in a finished dataset. In production the model runs at the wall,
and a column is only usable if its value would have been known **on the left-hand side of that line**.
The check is not "is this column in the data" but **"at the moment of prediction, had this happened
yet?"**

## Failure lab: two columns that cross the wall

Build two feature tables. The first uses only months 1 to 12. The second adds two columns that a
reasonable person might well write without noticing anything - a member's **lifetime visits** and their
**months on file** - both computed over the whole table, as such columns usually are.

**Predict before running:** how much better will the second model look?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

known_at_cut = pd.DataFrame({
    "visits_month_12": history[history.month == CUT].set_index("member_id").visits.reindex(active_at_cut),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active_at_cut),
    "tenure_months": history.groupby("member_id").size().reindex(active_at_cut),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active_at_cut),
})

whole_history = panel[panel.member_id.isin(active_at_cut)]
crosses_the_wall = known_at_cut.copy()
crosses_the_wall["visits_lifetime"] = whole_history.groupby("member_id").visits.sum().reindex(active_at_cut)
crosses_the_wall["months_on_file"] = whole_history.groupby("member_id").size().reindex(active_at_cut)


def score(features):
    train_X, test_X, train_y, test_y = train_test_split(
        features, target, test_size=0.3, random_state=0, stratify=target)
    centre, spread = train_X.mean(), train_X.std()
    model = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
    probability = model.predict_proba((test_X - centre) / spread)[:, 1]
    return roc_auc_score(test_y, probability), accuracy_score(test_y, probability > 0.5), test_y


auc_honest, acc_honest, held_out = score(known_at_cut)
auc_leaky, acc_leaky, _ = score(crosses_the_wall)

print("%-34s %8s %10s" % ("", "AUC", "accuracy"))
print("%-34s %8.3f %10.3f" % ("only what is known at month 12", auc_honest, acc_honest))
print("%-34s %8.3f %10.3f" % ("plus two whole-history columns", auc_leaky, acc_leaky))
print("%-34s %8.3f %10.3f" % ("predicting 'nobody leaves'", 0.500, 1 - held_out.mean()))

**AUC 1.000. A perfect model.** It would be presented on Thursday, approved, and deployed, and in
production it would score around 0.636, because the two extra columns cannot be computed for a member
whose future has not happened yet.

Look at why, because the mechanism is not subtle once you see it and is invisible until you do.

In [ ]:
for name in ["visits_month_12", "tenure_months", "visits_lifetime", "months_on_file"]:
    column = crosses_the_wall[name]
    print("%-18s correlation with the target %+.4f" % (name, np.corrcoef(column, target)[0, 1]))
print()
print("months on file, by outcome:")
print(pd.DataFrame({"months_on_file": crosses_the_wall.months_on_file, "cancelled_13_to_18": target})
      .groupby("cancelled_13_to_18").months_on_file.describe()[["count", "mean", "min", "max"]]
      .round(2).to_string())

`tenure_months`, computed honestly up to the wall, correlates **+0.0367** with the target - almost
nothing. `months_on_file`, the *same idea* computed over the whole table, correlates **-0.6051**.

The reason is arithmetic rather than statistics. A member's rows stop when they cancel, so the number of
rows they have *is* their cancellation month. **The column is a thinly disguised copy of the answer.**

But the interesting part is that it is not, by itself, enough to score perfectly - and the reason it
becomes enough is the part worth learning.

In [ ]:
last_month = whole_history.groupby("member_id").month.max().reindex(active_at_cut)
reconstructed = crosses_the_wall.months_on_file + CUT - crosses_the_wall.tenure_months

print("months_on_file + 12 - tenure_months recovers the exact last month:",
      bool((reconstructed == last_month).all()))
print()
print("last month on file, members who cancelled in the window : %d to %d"
      % (last_month[target == 1].min(), last_month[target == 1].max()))
print("last month on file, members who did not                 : %d to %d"
      % (last_month[target == 0].min(), last_month[target == 0].max()))


def auc_of(**columns):
    features = pd.DataFrame(columns)
    train_X, test_X, train_y, test_y = train_test_split(
        features, target, test_size=0.3, random_state=0, stratify=target)
    centre, spread = train_X.mean(), train_X.std()
    model = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
    return roc_auc_score(test_y, model.predict_proba((test_X - centre) / spread)[:, 1])


print()
print("AUC from tenure_months alone (honest)      : %.3f" % auc_of(t=crosses_the_wall.tenure_months))
print("AUC from months_on_file alone (leaky)      : %.3f" % auc_of(m=crosses_the_wall.months_on_file))
print("AUC from the two together                  : %.3f"
      % auc_of(m=crosses_the_wall.months_on_file, t=crosses_the_wall.tenure_months))

**Neither column is sufficient alone, and together they are exact.**

`tenure_months` is `13 - join_month`, so it encodes when the member joined - harmless on its own, and
correlated +0.0367 with the target. `months_on_file` is `last_month - join_month + 1`. Add the first to
the second and the join month cancels: **`months_on_file + 12 - tenure_months` is precisely the month
the membership ended.** Members who cancelled in the window end between months 13 and 18; everybody else
ends between 19 and 24. The two groups do not overlap at all, so a straight line separates them
perfectly and the AUC is exactly 1.000.

On its own `months_on_file` manages 0.954 - very suspicious, but not obviously impossible - because
members who joined late are short-tenured without having cancelled. It is the *harmless* column that
removes that confusion and completes the leak.

**So leakage is a property of the feature set, not only of individual columns.** Auditing features one at
a time would have cleared `tenure_months` correctly and might well have talked itself into keeping
`months_on_file` at 0.954. The reliable check is not per-column suspicion; it is the wall.

That is what leakage looks like from the inside: not a mistake in the modelling, but **values that could
only have been computed after the thing you are predicting had already happened**, sometimes only in
combination.

### The tell, and the habit

The tell is the score. **A model that is dramatically better than the problem should allow is reporting a
bug, not a result.** Churn from four behavioural columns does not give AUC 1.000; nothing does. The
correct reaction to a wonderful number is suspicion, and the correct next action is to ask of the
strongest feature: *when is this value known?*

The habit that prevents it: **compute every feature from a filtered table.** Not `panel.groupby(...)` but
`panel[panel.month <= CUT].groupby(...)`. Do the filtering once, at the top, and give the result a name
like `history` so that the wall exists in your code and not only in your head. Chapter 04-05 is a whole
lab on the four ways this goes wrong; the defence is set up here, in the framing.

## The honest model is worse than doing nothing. Also a framing problem.

Look again at the first row of that table: AUC **0.636**, accuracy **0.847**. Predicting that nobody
leaves scores **0.866**.

The honest model is **less accurate than a constant**, and it is still the more useful of the two, because
the AUC says it ranks members better than chance while the constant ranks nothing at all. Accuracy is the
wrong measure for a 13% base rate, and no amount of modelling fixes a badly chosen metric.

**That is a framing failure, not a modelling failure**, and it is why "what does good look like, in a
number, before we start?" belongs in the contract. Chapters 04-02 and 06-04 take this apart properly. For
now: if you cannot say what score would make the project worth doing, you cannot tell whether you have
succeeded.

## The contract

Five questions, five written answers. Here is the one this chapter arrived at.

| # | Question | Answer for this problem |
|---|---|---|
| 1 | **Unit of observation** | One active member, evaluated at a fixed month-end |
| 2 | **Target** | 1 if the member cancels within the horizon, else 0 - built by the line of code above, not by a description |
| 3 | **Prediction time** | End of month 12; in production, each month-end |
| 4 | **Horizon** | 6 months. Chosen because the retention campaign runs twice a year |
| 5 | **Available at prediction time** | Only rows with `month <= 12`. All features computed from `history`, never from `panel` |

Three more lines that cost nothing now and settle arguments later:

| | |
|---|---|
| **Population** | Members active at the prediction point. Already-cancelled members are excluded, not labelled 0 |
| **What good looks like** | Better ranking than chance by a margin worth the campaign cost - **not** accuracy, which a constant wins here at 86.6% |
| **The decision it feeds** | A retention offer to the top N members, where N is what the budget funds |

That last row is the one people leave out, and it is the one that makes the other seven answerable. **The
horizon came from the campaign schedule. The metric came from the fact that the budget only funds N
offers. The population came from the fact that you cannot make an offer to somebody who has left.** None
of that was in the data.

**A model is a means to a decision. If nobody can name the decision, the framing is not finished - and
that is a finding worth reporting, not a reason to start modelling anyway.**

## Common misconceptions

**"Framing is project-management paperwork."**
It selected the rows. Members who had already cancelled were dropped, months 19-24 were dropped, and the
target column was constructed. That is the dataset, and no modelling choice later can undo it.

**"The unit of observation is obvious from the request."**
"Predict which members will churn" is compatible with one row per member and one row per member-month,
and the two give 33.00% and 2.19% for the same phrase.

**"We can decide the horizon after we see how the model does."**
Then you have chosen the horizon that flatters the model. The base rate moves from 1.73% to 22.84% across
plausible horizons, so a horizon picked after the fact is a metric picked after the fact.

**"A column is safe if it is in the training data."**
Everything is in the training data. The question is whether it would exist at the moment of prediction,
and `months_on_file` would not.

**"A very high score means the model is good."**
It usually means the answer is in the features. AUC 1.000 on a behavioural churn problem is a bug report.

**"Accuracy is a neutral default."**
At a 13% base rate a constant scores 86.6% and the useful model scores 84.7%. The default punished the
better model.

**"If the framing turns out wrong we can refit later."**
Refitting is cheap. Re-collecting the labels for a different horizon, re-running the feature build behind
a different wall, and re-doing the evaluation is not - and by then the model is in production and
somebody is acting on it.

## Exercises

Solutions: `solutions/04_workflow/04-01_framing_solutions.ipynb`.

### Quick understanding

**E1.** Name the five framing questions from memory.

**E2.** In one sentence each, say what "unit of observation" decides and what "horizon" decides.

**E3.** Why were members who had already cancelled by month 12 excluded rather than labelled 0?

### Hand calculation

**E4.** A gym has 1,000 members. Over one year, 250 of them cancel. Memberships that survive the year
last 12 months; the ones that cancel last 6 on average. Compute (a) the per-member annual churn rate and
(b) the per-member-month churn rate. Show why they differ, using the number of member-months in the
denominator.

**E5.** Using the horizon table in this chapter, compute the accuracy of "nobody leaves" at horizons of
2 and 9 months. Then state the accuracy a model would need at each horizon in order to have demonstrated
anything at all.

**E6.** 521 members, 70 of whom cancel within the horizon. The retention budget funds 60 offers. If your
model's top 60 contains 22 real cancellers, what fraction of the offers are wasted, and what fraction of
the cancellers are reached? Compute both, and say which one the finance team will ask about.

**E7.** A member joins in month 3 and cancels in month 15. Give their value of `months_on_file` and of
`tenure_months` at a wall of month 12. Explain in one sentence why only one of the two is usable.

### Coding

**E8.** Write `frame(panel, cut, horizon)` returning `(features, target)` for any wall and horizon, with
every feature computed from rows at or before `cut`. Confirm it reproduces this chapter's 521 members and
13.44% base rate at `cut=12, horizon=6`.

**E9.** Use it to build the problem at `cut=6` and at `cut=18`. Report the population size and base rate
for each, and say which of the three walls you would use for a model that must be retrained monthly.

**E10.** Move the horizon from 1 to 12 with the wall fixed at 12, fit the honest model at each, and plot
AUC against horizon. Does the model get better or worse as the horizon grows? Give the mechanism, not
just the direction.

**E11.** Write `crosses_the_wall(panel, features, cut)` that flags any feature whose value changes when
recomputed on `panel[panel.month <= cut]` instead of the whole panel. Run it on `crosses_the_wall` and
confirm it catches exactly the two bad columns.

### Interpretation

**E12.** A colleague reports 94% accuracy on a churn model and asks to deploy it. List, in order, the
four questions you would ask before looking at any code.

**E13.** A vendor claims their churn model achieves "an industry-leading 91% accuracy". Explain what you
would need to know before that number means anything, and give a plausible way it could be true and
worthless simultaneously.

### Debugging

**E14.** A model scores AUC 0.998 in development and 0.61 in production. The code has not changed and the
data source has not changed. Give the two most likely causes and say how you would tell them apart.

### Exam and interview reasoning

**E15.** "How would you frame a churn prediction problem?" Answer in under two minutes, covering all five
questions and ending with the decision the model feeds. Then answer the follow-up: "why not just predict
whether they will ever leave?"

### Transfer to a different situation

**E16.** A hospital asks you to "predict which patients will be readmitted". Write the full contract -
all five questions plus population, metric and decision. Name one column that would certainly cross the
wall in that setting.

### Explain it to someone non-technical

**E17.** Explain to the head of retention, in under 100 words and without the words *target*, *feature*
or *leakage*, why you need to know when the campaign runs before you can build the model.

### Optional challenge

**E18.** Build the member-month version of this problem: one row per active member-month, with the target
"cancels within 6 months of *this* month", features computed from that member's history up to that month
only. Report the number of rows and the base rate, and name the new problem this creates that the
one-row-per-member version did not have. (Hint: the same member now appears many times.)

In [ ]:
# Your workspace. In memory: panel, CUT, HORIZON, active_at_cut, history, future,
# target, known_at_cut, crosses_the_wall, score, tenure.

## Mastery check

- [ ] State the five framing questions without looking
- [ ] Compute two defensible churn rates from the same table and explain why they differ
- [ ] Draw the wall picture for a problem you have been handed
- [ ] Say what a horizon changes: the population, the base rate, the metric and the useful features
- [ ] Test a column by asking when its value becomes known, not whether it is in the table
- [ ] Recognise a suspiciously good score as a bug report
- [ ] Say why accuracy at a 13% base rate rewarded the useless model
- [ ] Name the decision a model feeds, and say what to do when nobody can

## What should now feel instinctive

- Asking "what is one row?" before anything else, and refusing to accept a rate without its denominator
- Drawing the timeline before writing any feature code
- Building features from a filtered table with a name like `history`, so the wall lives in the code
- Treating a wonderful score as a symptom
- Asking what happens to the prediction after it is made, and stopping if there is no answer

## Flashcards

| Front | Back |
|---|---|
| Framing | Turning a request into a task: five questions with written answers |
| The five questions | Unit, target, prediction time, horizon, availability |
| Unit of observation | What one row is. It sets every denominator and every average |
| Same word, three rates here | 33.00% per member, 2.19% per member-month, 1.51% in month 12 |
| Prediction time | The wall. Left of it is features, right of it is the answer |
| Horizon | How far past the wall the target looks. Moved the base rate 1.73% to 22.84% |
| Availability test | Would this value have been known at the wall? Not: is it in the table |
| Leakage tell | A score better than the problem allows. AUC 1.000 here |
| `months_on_file` | 15 rows means cancelled in month 15. A copy of the answer, correlation -0.61 |
| Population | Who is eligible for a prediction. Already-cancelled members are excluded, not labelled 0 |
| Accuracy at a 13% base rate | A constant scores 86.6%; the useful model scored 84.7% |
| The last question | What decision does this feed? If nobody can say, the framing is not finished |

## Next

**04-02 · Baselines first, always.** This chapter ended on a number that should be uncomfortable: a
constant prediction scored 86.6% and beat a model that had genuinely learned something. That was not bad
luck, and it is not rare - it is the normal condition of an imbalanced problem, and the only defence is to
compute what "doing nothing" scores **before** you fit anything.

The next chapter builds those baselines properly - the constant, the last-value-carried-forward, the
one-rule model - and shows how to state a result as *skill over a baseline* rather than as a number that
sounds impressive on its own.